# Instalment Payments – Feature Engineering

Step 1 of preprocessing for `installments_payments.csv`: the repayment
history of the applicant's previous Home Credit loans. One row per
instalment payment: when it was due, when it was actually paid, how much
was due and how much was paid.

Needs `prev_loan_data_processed.parquet` from `Prevloans_data.ipynb` first,
for details of each previous loan.

**Sections**
1–2. Imports and settings
3. Load, keeping only applicants in the training data
4. Work out each payment's month, and sort
5. Fill in missing payment dates and amounts
6. Per-payment features (days late, share paid, schedule changes)
7. One row per applicant per month
8. Per-loan summaries, then per applicant (all time, last year, last 6 months)
9. Details from the previous-application data
10. Summaries over recent periods (3 years … 3 months)
11. The latest month's values
12. Combine everything, plus recent-vs-long-term ratios
13. Quality checks
14. Save
15. Free memory

**Output:** `data/Model-Training/Staging/payment_installments_processed.parquet`
— one row per applicant, columns prefixed `PMNT_INSTLMT_`.

Run by the `01_preprocessing` Airflow DAG (task `task_installments`, after
`task_prevloans`), or by hand.

## 1. Imports

In [1]:
import os
import pandas as pd
import numpy as np
import gc

## 2. Configuration & Paths

`DATA_ROOT` is the project's `data/` folder: `/data` inside Docker,
otherwise this project's folder on your PC.

In [2]:
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

PATH_PAYMENTS    = os.path.join(DATA_ROOT, "Input", "installments_payments.csv")
PATH_APP_TRAIN   = os.path.join(DATA_ROOT, "Input", "application_train.csv")
PATH_PREV_LOANS  = os.path.join(DATA_ROOT, "Model-Training", "Staging", "prev_loan_data_processed.parquet")   # from Prevloans_data.ipynb
PATH_OUTPUT      = os.path.join(DATA_ROOT, "Model-Training", "Staging", "payment_installments_processed.parquet")

## 3. Load Data

Only applicants in `application_train.csv` are kept. `TARGET` is read with
the IDs but dropped straight after the join, so it can't leak into the
features. The list of applicants is kept as the base for the final table,
so nobody is lost if they're missing from some of the summaries.

In [3]:
base = pd.read_csv(PATH_APP_TRAIN, usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

Payments = pd.read_csv(PATH_PAYMENTS)
Payments = Payments.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

# Keep only training applicants; drop TARGET straight away
Payments = base.merge(Payments, on='SK_ID_CURR', how='inner').drop(columns=['TARGET'])

# Every applicant with repayment history -- the base the final table is built on (section 12)
unique_customers = Payments[['SK_ID_CURR']].drop_duplicates()

print(f"Payments shape: {Payments.shape}")

Payments shape: (11591592, 8)


## 4. Work Out Each Payment's Month, and Sort

`DAYS_INSTALMENT` (the due date) is days before the application, as a
negative number. Dividing by 30 and rounding **down** gives the month
(−1 = last month). Rounding down rather than towards zero matters for
negative numbers: −12.9 becomes −13, keeping each month's days together.

Rows are sorted by applicant, loan, month and actual payment date, so
"first" and "last" below mean the earliest and latest payment.

In [4]:
# Due date (days) -> month, rounding down (so -12.9 -> -13, not -12)
Payments['MONTHS_BALANCE'] = np.floor(Payments['DAYS_INSTALMENT'] / 30).astype(int)

Payments.sort_values(
    by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE', 'DAYS_ENTRY_PAYMENT'],
    inplace=True,
    ignore_index=True,
)

print(f"MONTHS_BALANCE range: [{Payments['MONTHS_BALANCE'].min()}, {Payments['MONTHS_BALANCE'].max()}]")

MONTHS_BALANCE range: [-98, -1]


## 5. Fill in Missing Payment Dates and Amounts

A missing payment date or amount is filled from the nearest payment on the
same loan (earlier payments first), up to 6 rows away; longer gaps stay missing.

In [5]:
for col in ['DAYS_ENTRY_PAYMENT', 'AMT_PAYMENT']:
    Payments[col] = (
        Payments.groupby('SK_ID_PREV')[col]
        .transform(lambda x: x.ffill(limit=6).bfill(limit=6))
    )

print("Remaining nulls after imputation:")
print(Payments[['DAYS_ENTRY_PAYMENT', 'AMT_PAYMENT']].isna().sum().to_string())

Remaining nulls after imputation:
DAYS_ENTRY_PAYMENT    508
AMT_PAYMENT           508


## 6. Per-Payment Features

- `LATE_PAYMENT_DAYS` — days paid after the due date (negative = paid early).
- `PAYMENT_AMT_TO_INSTALLMENT_RATIO` — amount paid ÷ amount due (1 = paid in full).
- `LATE_PAYMENT_FLAG` — 1 if paid late.
- `INSTALMENT_VERSION_CHANGE` — 1 (or more) when the repayment schedule
  version changed from the previous payment, e.g. the loan was renegotiated.

In [6]:
Payments['LATE_PAYMENT_DAYS'] = (
    Payments['DAYS_ENTRY_PAYMENT'] - Payments['DAYS_INSTALMENT']
)
Payments['PAYMENT_AMT_TO_INSTALLMENT_RATIO'] = np.where(
    Payments['AMT_INSTALMENT'] > 0,
    Payments['AMT_PAYMENT'] / Payments['AMT_INSTALMENT'],
    np.nan,
)
Payments['LATE_PAYMENT_FLAG'] = (
    Payments['LATE_PAYMENT_DAYS'] > 0
).astype(int)

# Change in the schedule version since the previous payment on the same loan (0 = unchanged)
Payments['INSTALMENT_VERSION_CHANGE'] = (
    Payments.groupby('SK_ID_PREV')['NUM_INSTALMENT_VERSION']
    .diff()
    .fillna(0)
    .astype(int)
)

print("Late payment rate:",
      f"{Payments['LATE_PAYMENT_FLAG'].mean():.2%}")

Late payment rate: 8.57%


## 7. One Row per Applicant per Month

Amounts paid and due (and schedule changes) are **added up** across the
applicant's loans for the month; lateness and share paid take the month's
**highest** value. Columns are prefixed `PMNT_INSTLMT_`.

In [7]:
sum_cols = ['AMT_PAYMENT', 'AMT_INSTALMENT', 'INSTALMENT_VERSION_CHANGE']           # added up per month
max_cols  = ['LATE_PAYMENT_DAYS', 'PAYMENT_AMT_TO_INSTALLMENT_RATIO', 'LATE_PAYMENT_FLAG']   # highest per month

Payments_Mon_Sum = (
    Payments.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[sum_cols]
    .sum()
    .reset_index()
)
Payments_Mon_Sum.columns = [
    'PMNT_INSTLMT_' + col if col not in ('SK_ID_CURR', 'MONTHS_BALANCE') else col
    for col in Payments_Mon_Sum.columns
]

Payments_Mon_Max = (
    Payments.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[max_cols]
    .max()
    .reset_index()
)
Payments_Mon_Max.columns = [
    'PMNT_INSTLMT_' + col if col not in ('SK_ID_CURR', 'MONTHS_BALANCE') else col
    for col in Payments_Mon_Max.columns
]

Payments_Mon = Payments_Mon_Sum.merge(
    Payments_Mon_Max, on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left'
)
del Payments_Mon_Sum, Payments_Mon_Max

print(f"Payments_Mon shape: {Payments_Mon.shape}")

Payments_Mon shape: (8199221, 8)


## 8. Per-Loan Summaries, then per Applicant

First one row per previous loan (`SK_ID_PREV`):

- `VRSN_CHANGES_COUNT` — how many different schedule versions the loan had
- `DURATION_DAYS` — days from the first to the last payment
- `AVG_INSTALLMENT_FREQ` — average days between payments (using distinct
  payment dates, sorted, so the result can't be negative)
- `NUM_INSTALMENTS` — number of payments

Then the mean and max of each across the applicant's loans — over all
time, and (for schedule versions and payment frequency) over the last year
and last 6 months.

In [8]:
# ── One row per loan ──────────────────────────────────────────────────────
Payments_Agg_PrevID = (
    Payments.groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(
        PMNT_INSTLMT_VRSN_CHANGES_COUNT=('NUM_INSTALMENT_VERSION', 'nunique'),
        PMNT_INSTLMT_DURATION_DAYS=('DAYS_ENTRY_PAYMENT', lambda x: x.max() - x.min() + 1),
        # Average gap in days between distinct payment dates
        PMNT_INSTLMT_AVG_INSTALLMENT_FREQ=('DAYS_ENTRY_PAYMENT',
            lambda x: (
                pd.Series(x.unique()).sort_values().diff().mean()
                if len(x) > 1 else np.nan
            ),
        ),
        PMNT_INSTLMT_NUM_INSTALMENTS=('NUM_INSTALMENT_NUMBER', 'count'),
    )
    .reset_index()
)

# ── Per applicant, all time: mean and max across their loans ──────────────
Payments_Agg_PrevID_counts = (
    Payments_Agg_PrevID
    .groupby('SK_ID_CURR')
    .agg({
        'PMNT_INSTLMT_VRSN_CHANGES_COUNT': ['mean', 'max'],
        'PMNT_INSTLMT_DURATION_DAYS': ['mean', 'max'],
        'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ': ['mean', 'max'],
        'PMNT_INSTLMT_NUM_INSTALMENTS': ['mean', 'max'],
    })
    .reset_index()  # make SK_ID_CURR a normal column again
)
# Two-level names like (column, 'mean') -> COLUMN_MEAN
Payments_Agg_PrevID_counts.columns = [
    '_'.join(col).strip().upper() if col[1] else col[0]
    for col in Payments_Agg_PrevID_counts.columns
]

# ── Per applicant, last year (payments due in the last 12 months) ─────────
Payments_Agg_PrevID_Last1Y = (
    Payments[Payments['MONTHS_BALANCE'] >= -12]
    .groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(
        PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST1Y=('NUM_INSTALMENT_VERSION', 'nunique'),
        PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_LAST1Y=('DAYS_ENTRY_PAYMENT', lambda x: ( pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan ),
        ),
    )
    .reset_index()
)
Payments_Agg_PrevID_Last1Y_counts = (
    Payments_Agg_PrevID_Last1Y
    .groupby('SK_ID_CURR')
    .agg({
        'PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST1Y':  ['mean', 'max'],
        'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_LAST1Y': ['mean', 'max'],
    })
    .reset_index()  # make SK_ID_CURR a normal column again
)
Payments_Agg_PrevID_Last1Y_counts.columns = [
    '_'.join(col).strip().upper() if col[1] else col[0]
    for col in Payments_Agg_PrevID_Last1Y_counts.columns
]

# ── Per applicant, last 6 months ──────────────────────────────────────────
Payments_Agg_PrevID_Last6M = (
    Payments[Payments['MONTHS_BALANCE'] >= -6]
    .groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(
        PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST6M=('NUM_INSTALMENT_VERSION', 'nunique'),
        PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_LAST6M=('DAYS_ENTRY_PAYMENT', lambda x: (pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan ),
        ),
    )
    .reset_index()
)
Payments_Agg_PrevID_Last6M_counts = (
    Payments_Agg_PrevID_Last6M
    .groupby('SK_ID_CURR')
    .agg({
        'PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST6M':  ['mean', 'max'],
        'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_LAST6M': ['mean', 'max'],
    })
    .reset_index()  # make SK_ID_CURR a normal column again
)
Payments_Agg_PrevID_Last6M_counts.columns = [
    '_'.join(col).strip().upper() if col[1] else col[0]
    for col in Payments_Agg_PrevID_Last6M_counts.columns
]

print(f"All-time rollup shape : {Payments_Agg_PrevID_counts.shape}")
print(f"Last-1Y rollup shape  : {Payments_Agg_PrevID_Last1Y_counts.shape}")
print(f"Last-6M rollup shape  : {Payments_Agg_PrevID_Last6M_counts.shape}")

All-time rollup shape : (291643, 9)
Last-1Y rollup shape  : (216759, 5)
Last-6M rollup shape  : (182254, 5)


In [9]:
del Payments_Agg_PrevID_Last1Y, Payments_Agg_PrevID_Last6M
gc.collect()

0

## 9. Details from the Previous-Application Data

Adds each loan's details from `prev_loan_data_processed.parquet`, then
summarises per applicant: number of loans, average amount granted ÷ amount
asked for, average repayment ÷ amount, how many were in the high/middle
interest-rate groups, how many were insured, and how long ago they were
decided on average.

In [10]:
PrevLoans = pd.read_parquet(PATH_PREV_LOANS)
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

Payments_PrevLoans = Payments_Agg_PrevID.merge(
    PrevLoans, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left'
)

Payments_Agg_PrevLoans = (
    Payments_PrevLoans.groupby('SK_ID_CURR')
    .agg(
        PMNT_INSTLMT_TOTAL_PREV_LOAN=('SK_ID_PREV', 'nunique'),
        PMNT_INSTLMT_AVG_CREDIT_TO_APPL=('CREDIT_TO_APPL', 'mean'),
        PMNT_INSTLMT_ANNUITY_TO_CREDIT_RATIO=('AMT_ANNUITY_TO_CREDIT_RATIO', 'mean'),
        PMNT_INSTLMT_CNT_HIGH_YIELD_GROUP=('NAME_YIELD_GROUP', lambda x: (x == 'high').sum()),
        PMNT_INSTLMT_CNT_MIDDLE_YIELD_GROUP=('NAME_YIELD_GROUP', lambda x: (x == 'middle').sum()),
        PMNT_INSTLMT_CNT_NFLAG_INSURED_ON_APPROVAL=('NFLAG_INSURED_ON_APPROVAL', 'sum'),
        PMNT_INSTLMT_AVG_DAYS_DECISION=('DAYS_DECISION', 'mean'),
    )
    .reset_index() 
)

del Payments_Agg_PrevID, Payments_PrevLoans, PrevLoans
print(f"PrevLoans enrichment shape: {Payments_Agg_PrevLoans.shape}")

PrevLoans enrichment shape: (291643, 8)


## 10. Summaries over Recent Periods

From the monthly table (section 7): the mean, max and min of each monthly
column over the last 3 years, 2 years, 1 year, 6 months and 3 months.
Columns are named like `PMNT_INSTLMT_LATE_PAYMENT_DAYS_MAX_LAST1Y`.

The current @prod model uses three of these: `…LATE_PAYMENT_DAYS_MAX_LAST1Y`,
`…LATE_PAYMENT_DAYS_MEAN_LAST6M` and `…PAYMENT_AMT_TO_INSTALLMENT_RATIO_MEAN_LAST2Y`.

In [11]:
agg_cols = [
    'PMNT_INSTLMT_AMT_PAYMENT',
    'PMNT_INSTLMT_AMT_INSTALMENT',
    'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE',
    'PMNT_INSTLMT_LATE_PAYMENT_DAYS',
    'PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO',
    'PMNT_INSTLMT_LATE_PAYMENT_FLAG',
]

# Each block: keep months in the period, then mean/max/min per applicant.
# Two-level names like (column, 'mean') become COLUMN_MEAN_<PERIOD>.

# ── Last 3 years ──────────────────────────────────────────────────────────
Payments_Mon_3Y_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -36]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()
)
Payments_Mon_3Y_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST3Y' if col[1] else col[0]
    for col in Payments_Mon_3Y_Agg.columns
]

# ── Last 2 years ──────────────────────────────────────────────────────────
Payments_Mon_2Y_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -24]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()
)
Payments_Mon_2Y_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST2Y' if col[1] else col[0]
    for col in Payments_Mon_2Y_Agg.columns
]

# ── Last year ─────────────────────────────────────────────────────────────
Payments_Mon_1Y_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -12]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()
)
Payments_Mon_1Y_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST1Y' if col[1] else col[0]
    for col in Payments_Mon_1Y_Agg.columns
]

# ── Last 6 months ─────────────────────────────────────────────────────────
Payments_Mon_6M_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -6]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()
)
Payments_Mon_6M_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST6M' if col[1] else col[0]
    for col in Payments_Mon_6M_Agg.columns
]

# ── Last 3 months ─────────────────────────────────────────────────────────
Payments_Mon_3M_Agg = (
    Payments_Mon[Payments_Mon['MONTHS_BALANCE'] >= -3]
    .groupby('SK_ID_CURR')[agg_cols]
    .agg(['mean', 'max', 'min'])
    .reset_index()
)
Payments_Mon_3M_Agg.columns = [
    '_'.join(col).strip().upper() + '_LAST3M' if col[1] else col[0]
    for col in Payments_Mon_3M_Agg.columns
]

print(f"3Y agg shape: {Payments_Mon_3Y_Agg.shape}")
print(f"2Y agg shape: {Payments_Mon_2Y_Agg.shape}")
print(f"1Y agg shape: {Payments_Mon_1Y_Agg.shape}")
print(f"6M agg shape: {Payments_Mon_6M_Agg.shape}")
print(f"3M agg shape: {Payments_Mon_3M_Agg.shape}")

3Y agg shape: (264817, 19)
2Y agg shape: (251476, 19)
1Y agg shape: (216759, 19)
6M agg shape: (182254, 19)
3M agg shape: (159944, 19)


## 11. The Latest Month's Values

Each applicant's most recent month in the monthly table, with every column
suffixed `_LAST_MONTH`. It uses `.nth(-1)` (the actual last row) rather than
`.last()`, which skips blanks and could return an older month's value.

`PMNT_INSTALMENT_MONTHS_SINCE_LAST_TRANSACTION` = how many months ago that
latest month was (the sign is flipped so it reads as a positive "months ago").

In [12]:
# The actual latest month per applicant (.nth(-1); .last() would skip blanks)
Payments_Last_Snapshot = (
    Payments_Mon.sort_values('MONTHS_BALANCE')
    .groupby('SK_ID_CURR')[['SK_ID_CURR','MONTHS_BALANCE'] + agg_cols]
    .nth(-1)
    .reset_index()
)
Payments_Last_Snapshot.columns = [
    col + '_LAST_MONTH' if col != 'SK_ID_CURR' else col
    for col in Payments_Last_Snapshot.columns
]
Payments_Last_Snapshot.rename(
    columns={'MONTHS_BALANCE_LAST_MONTH': 'PMNT_INSTALMENT_MONTHS_SINCE_LAST_TRANSACTION'},
    inplace=True,
)
# Flip the sign so it reads as a positive "months ago"
Payments_Last_Snapshot['PMNT_INSTALMENT_MONTHS_SINCE_LAST_TRANSACTION'] = (
    -Payments_Last_Snapshot['PMNT_INSTALMENT_MONTHS_SINCE_LAST_TRANSACTION']
)

del Payments_Mon
print(f"Last-snapshot shape: {Payments_Last_Snapshot.shape}")

Last-snapshot shape: (291643, 9)


## 12. Combine Everything

Starts from the full list of applicants saved at load time, so nobody is
dropped, and joins on every summary — one row per applicant. For each
recent period a yes/no flag (`PMNT_INSTLMT_IS_IN_LAST_3Y` …
`PMNT_INSTLMT_IS_IN_LAST_3M`) says whether the applicant had any payment due in it.

In [13]:
def flag_merge(left, right, on, flag_col):
    """Left-join `right` onto `left` and add `flag_col`: 1 if the row had a match in `right`, else 0."""
    merged = left.merge(right, on=on, how='left', indicator=True)
    merged[flag_col] = (merged['_merge'] == 'both').astype(int)
    return merged.drop(columns=['_merge'])


# Start from every applicant with repayment history
Payments_Agg_final = unique_customers.copy()

Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Agg_PrevID_counts, on='SK_ID_CURR', how='left'
)
Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Agg_PrevID_Last1Y_counts, on='SK_ID_CURR', how='left'
)
Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Agg_PrevID_Last6M_counts, on='SK_ID_CURR', how='left'
)
# Recent-period summaries, each with a "had payments in this period" flag
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_3Y_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_3Y'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_2Y_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_2Y'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_1Y_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_1Y'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_6M_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_6M'
)
Payments_Agg_final = flag_merge(
    Payments_Agg_final, Payments_Mon_3M_Agg, 'SK_ID_CURR', 'PMNT_INSTLMT_IS_IN_LAST_3M'
)
Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Last_Snapshot, on='SK_ID_CURR', how='left'
)
Payments_Agg_final = Payments_Agg_final.merge(
    Payments_Agg_PrevLoans, on='SK_ID_CURR', how='left'
)

print(f"Final shape: {Payments_Agg_final.shape}")
Payments_Agg_final.head(3)

Final shape: (291643, 127)


,SK_ID_CURR,PMNT_INSTLMT_VRSN_CHANGES_COUNT_MEAN,PMNT_INSTLMT_VRSN_CHANGES_COUNT_MAX,PMNT_INSTLMT_DURATION_DAYS_MEAN,PMNT_INSTLMT_DURATION_DAYS_MAX,PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_MEAN,PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_MAX,PMNT_INSTLMT_NUM_INSTALMENTS_MEAN,PMNT_INSTLMT_NUM_INSTALMENTS_MAX,PMNT_INSTLMT_VRSN_CHANGES_COUNT_LAST1Y_MEAN,...,PMNT_INSTLMT_LATE_PAYMENT_DAYS_LAST_MONTH,PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_LAST_MONTH,PMNT_INSTLMT_LATE_PAYMENT_FLAG_LAST_MONTH,PMNT_INSTLMT_TOTAL_PREV_LOAN,PMNT_INSTLMT_AVG_CREDIT_TO_APPL,PMNT_INSTLMT_ANNUITY_TO_CREDIT_RATIO,PMNT_INSTLMT_CNT_HIGH_YIELD_GROUP,PMNT_INSTLMT_CNT_MIDDLE_YIELD_GROUP,PMNT_INSTLMT_CNT_NFLAG_INSURED_ON_APPROVAL,PMNT_INSTLMT_AVG_DAYS_DECISION
0,100002,2.000000,2,539.000000,539.0,29.888889,29.888889,19.000000,19,2.0,...,-24.0,1.0,0,1,1.000000,0.051670,0,0,0.0,-606.0
1,100003,1.333333,2,220.666667,340.0,29.661616,30.818182,8.333333,12,NaN,...,-8.0,1.0,0,3,1.057664,0.126383,0,2,2.0,-1305.0
2,100004,2.000000,2,69.000000,69.0,34.000000,34.000000,3.000000,3,NaN,...,-3.0,1.0,0,1,0.828021,0.266450,0,1,0.0,-815.0


**Recent vs. long-term ratios:** each feature's last-6-months average ÷ its
last-3-years average. Above 1 means the value has risen recently — e.g. for
lateness, the applicant has been paying later lately than over the longer
term. Left blank when the 3-year value is 0.

In [18]:
def add_period_ratio_features(df, attributes, numerator_period='6M', denominator_period='3Y'):
    """
    For each attribute, add <attr>_LAST<numerator_period>_TO_<denominator_period>_RATIO
    = <attr>_LAST<numerator_period> ÷ <attr>_LAST<denominator_period>,
    left blank (NaN) when the denominator is 0.

    Both <attr>_LAST... columns must already exist in df (here they come
    from the recent-period summaries in section 10).
    """
    for attr in attributes:
        numerator_col = f'{attr}_LAST{numerator_period}'
        denominator_col = f'{attr}_LAST{denominator_period}'
        ratio_col = f'{attr}_LAST{numerator_period}_TO_{denominator_period}_RATIO'
        df[ratio_col] = np.where(
            df[denominator_col] != 0,
            df[numerator_col] / df[denominator_col],
            np.nan,
        )
    return df

In [22]:
recent_vs_historical_attrs = [
    'PMNT_INSTLMT_LATE_PAYMENT_FLAG_MEAN',
    'PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MEAN',
    'PMNT_INSTLMT_LATE_PAYMENT_DAYS_MEAN',
    'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE_MEAN',
    'PMNT_INSTLMT_AMT_INSTALMENT_MEAN',
    'PMNT_INSTLMT_AMT_PAYMENT_MEAN'
]

Payments_Agg_final = add_period_ratio_features(
    Payments_Agg_final, recent_vs_historical_attrs,
    numerator_period='6M', denominator_period='3Y',
)

## 13. Quality Checks

The share of missing values in the emptiest columns, and exactly one row per applicant.

In [23]:
null_rates = Payments_Agg_final.isna().mean().sort_values(ascending=False)
print("Null rate (top 15 columns):")
print(null_rates.head(15).to_string())

n_dupes = Payments_Agg_final['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR found: {n_dupes}"
print(f"\nNo duplicate SK_ID_CURR. Final row count: {len(Payments_Agg_final):,}")

Null rate (top 15 columns):
PMNT_INSTLMT_LATE_PAYMENT_FLAG_MEAN_LAST6M_TO_3Y_RATIO            0.725284
PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE_MEAN_LAST6M_TO_3Y_RATIO    0.634515
PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MIN_LAST3M          0.451693
PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MAX_LAST3M          0.451693
PMNT_INSTLMT_LATE_PAYMENT_DAYS_MIN_LAST3M                         0.451693
PMNT_INSTLMT_LATE_PAYMENT_DAYS_MEAN_LAST3M                        0.451693
PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MEAN_LAST3M         0.451693
PMNT_INSTLMT_LATE_PAYMENT_DAYS_MAX_LAST3M                         0.451693
PMNT_INSTLMT_LATE_PAYMENT_FLAG_MAX_LAST3M                         0.451576
PMNT_INSTLMT_AMT_INSTALMENT_MEAN_LAST3M                           0.451576
PMNT_INSTLMT_LATE_PAYMENT_FLAG_MIN_LAST3M                         0.451576
PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE_MIN_LAST3M                 0.451576
PMNT_INSTLMT_LATE_PAYMENT_FLAG_MEAN_LAST3M                        0.4515

## 14. Save

In [24]:
Payments_Agg_final.to_parquet(PATH_OUTPUT, index=False)
print(f"Saved → {PATH_OUTPUT}")

Saved → a:\Dominic\My Projects\Home Credit\data\Model-Training\Staging\payment_installments_processed.parquet


## 15. Free Memory

In [25]:
del (
    Payments, Payments_Agg_PrevID_counts, Payments_Agg_PrevID_Last1Y_counts,
    Payments_Mon_1Y_Agg, Payments_Mon_3M_Agg, Payments_Last_Snapshot,
    Payments_Agg_PrevLoans, unique_customers, base,
)
gc.collect()
print("All DataFrames released.")

All DataFrames released.
